# Linear Regression to Predict Titanic Passenger Age

**Deep Dive Coding Data Science Bootcamp Sample Project**

---
Robert Citek - October 5th, 2026


## Problem Definition

State the business problem. Translate the business problem into a Data Science problem by stating what kind of problem it is ( supervised vs unsupervised ) and whether it is a classification, regression, or clustering problem.  Mention which model will be used, if known.

### Regression

For a regression, the Age will be the target.


## Data Collection/Sources

### Data Overview

This data comes from [Kaggle](https://www.kaggle.com/c/titanic/data), which was then copied to AWS for easy access.

https://ddc-datascience.s3.amazonaws.com/Projects/Example/Data/Titanic.train.csv


### Imports

In [1]:
from IPython.display import Image
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns

from sklearn import datasets, metrics, model_selection
from sklearn import model_selection
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

### File Paths

Create a variable `url` that contains the web link to the CSV data

In [2]:
url = 'https://ddc-datascience.s3.amazonaws.com/Projects/Example/Data/Titanic.train.csv'    # insert url here
url

'https://ddc-datascience.s3.amazonaws.com/Projects/Example/Data/Titanic.train.csv'

### Load data

Load the data into a data frame named `titanic_df`.

In [3]:
# load data
titanic_df = pd.read_csv( url )

### head

Look at the first few rows

In [4]:
# first few rows
titanic_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### tail

Look at the last few rows


In [5]:
# last few rows
titanic_df.tail()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.00,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.00,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q


### shape


In [6]:
# shape of data frame
titanic_df.shape

(891, 12)

### size


In [7]:
# size of data frame
titanic_df.size

10692

### Summary meta-data

Run the `info()` method on the data frame.


In [8]:
# display meta-data
titanic_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


### Summary statistics

Summary statistics give you a quick overview of the data.  Some items to spot check:
- the Target:
  - look for nulls
    - null will be culled
  - look at min, max, median, and mean
    - is it a categorical field masquerading as a number?
- the Features:
  - numerical
    - look at min, max, median, and mean
      - a min=1 and max=count suggests an ID
  - categorical
    - look at unique and freq
      - freq=1, unique=count suggests an ID
      - freq=count suggests a meaningless feature

Run the `describe()` method on the data frame.

In [9]:
# display summary statistics
titanic_df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
PassengerId,891.0,446.000000,257.353842,1.00,223.5000,446.0000,668.5,891.0000
Survived,891.0,0.383838,0.486592,0.00,0.0000,0.0000,1.0,1.0000
Pclass,891.0,2.308642,0.836071,1.00,2.0000,3.0000,3.0,3.0000
Age,714.0,29.699118,14.526497,0.42,20.1250,28.0000,38.0,80.0000
SibSp,891.0,0.523008,1.102743,0.00,0.0000,0.0000,1.0,8.0000
Parch,891.0,0.381594,0.806057,0.00,0.0000,0.0000,0.0,6.0000
Fare,891.0,32.204208,49.693429,0.00,7.9104,14.4542,31.0,512.3292


### target nulls


The target should be the first field that you look at.  Its data type will tell you what kind of machine learning problem this is ( classification or regression. ) Any nulls should be culled.  You can check for any skew in the data set.

In [30]:
target_col = 'Age'
titanic_df[target_col]

,Age
0,22.0
1,38.0
2,26.0
3,35.0
4,35.0
...,...
886,27.0
887,19.0
888,NaN
889,26.0


In [31]:
# look for nulls in the target
titanic_df[target_col].isnull().sum() / titanic_df[target_col].shape[0] * 100
# I am comfortable dropping nulls if 20% or under

np.float64(19.865319865319865)

### Nulls



In [34]:
# look for nulls
titanic_df.drop(columns=[target_col]).isnull().sum().sort_values(ascending=False)
# Cabin and Embarked have nulls

,0
Cabin,687
Embarked,2
PassengerId,0
Pclass,0
Survived,0
Name,0
Sex,0
Parch,0
SibSp,0
Fare,0


In [36]:
# look for nulls, displaying relative frequency
(titanic_df.drop(columns=[target_col]).isnull().sum()/ titanic_df.shape[0] * 100).drop(columns=[target_col]).sort_values(ascending=False)
# cabin has 77% nulls and embarked has 0.2%

,0
Cabin,77.104377
Embarked,0.224467
PassengerId,0.000000
Pclass,0.000000
Survived,0.000000
Name,0.000000
Sex,0.000000
Parch,0.000000
SibSp,0.000000
Fare,0.000000


### dtypes - column data types

Which data types would be appropriate for Linear Regression?


In [38]:
# display data types
titanic_df.dtypes.sort_values(ascending=False)

,0
Name,object
Sex,object
Ticket,object
Cabin,object
Embarked,object
Age,float64
Fare,float64
PassengerId,int64
Survived,int64
Pclass,int64


### Unique elements


If the total number of unique items in a field equals the number of rows, that suggests those fields are unique IDs ( e.g. names, IDs ) and should be dropped.


In [42]:
# number of unique elements in a column
(titanic_df.nunique() / titanic_df.shape[0] * 100).sort_values(ascending=False)
# PassengerId and Name are 100% unique values so those are indeed identifier columns. Ticket is 76% unique values so probably not a useful feature.
# Age and Fare have low unique values so they are OK to work with.

,0
PassengerId,100.000000
Name,100.000000
Ticket,76.430976
Fare,27.833895
Cabin,16.498316
Age,9.876543
SibSp,0.785634
Parch,0.785634
Embarked,0.336700
Pclass,0.336700


## Data Cleaning


### Backup #1


In [43]:
# make backup
titanic_df_bak1 = titanic_df.copy()

In [44]:
# restore from backup
titanic_df = titanic_df_bak1.copy()

### Keep only the target and two features

Select `Age` for the target and `Fare` for the feature

In [45]:
# select the target and one feature
feature_col = 'Fare'

In [48]:
# Verify the target column
titanic_df[target_col]

,Age
0,22.0
1,38.0
2,26.0
3,35.0
4,35.0
...,...
886,27.0
887,19.0
888,NaN
889,26.0


In [49]:
# Verify the feature column
titanic_df[feature_col]

,Fare
0,7.2500
1,71.2833
2,7.9250
3,53.1000
4,8.0500
...,...
886,13.0000
887,30.0000
888,23.4500
889,30.0000


### Backup #2


In [50]:
# make backup
titanic_df_bak2 = titanic_df.copy()

In [51]:
# restore from backup
titanic_df = titanic_df_bak2.copy()

### Address any rows with nulls

- If the number of nulls in a column is low, it would be easiest to drop the row.
- If the number of nulls in a column is high, it would be easiest to drop the column.
- If the number of nulls in a column is between the low and high, one should impute values for the nulls.


In [55]:
# Modify the dataframe so it now only contains the Age (target) and Fare (feature) columns
titanic_df = titanic_df[[target_col, feature_col]]
titanic_df

,Age,Fare
0,22.0,7.2500
1,38.0,71.2833
2,26.0,7.9250
3,35.0,53.1000
4,35.0,8.0500
...,...,...
886,27.0,13.0000
887,19.0,30.0000
888,NaN,23.4500
889,26.0,30.0000


In [58]:
# Modify the dataframe to drop all rows across our dataframe that contain nulls when looking into our target column.
titanic_df = titanic_df.dropna(subset=target_col)

In [60]:
# Verify the shape
titanic_df.shape
# Shape now shows 714 rows down from 891 rows

(714, 2)

#### Impute

**No inputation needed. No nulls in our feature 'Fare' and we dropped all the null columns referencing our target 'Age' column.**

Impute with the median.


In [ ]:
# Calculate the median


In [ ]:
# Assign median to nulls


In [ ]:
# Verify there are no nulls


In [ ]:
# Verify median is about the same as before


## Exploratory Data Analysis


Linear Regression has two requirements:
1. the features are correlated with the target
1. the features are not correlated with each other

### correlated with the target


In [ ]:
# Calculate a correlation matrix for the target


In [ ]:
# Display the correlation matrix


In [ ]:
# Create a data series with features ranked by correlation


### independence of features: correlation

In [ ]:
# Calculate a correlation matrix for the features


In [ ]:
# Display the correlation matrix


In [ ]:
# Create a data series with features ranked by correlation


## Processing

### Prepare data frame

In [ ]:
# Split data frame into target and features


In [ ]:
# Verify features


In [ ]:
# Verify target


### Multiple Cross Validation ( CV )

Create a CV loop with 100 iterations.  Capture performance metric in an array names `results`.
Use accuracy for the performance metric.


In [ ]:
# Multiple CV loop


#### Performance

We'll use RMSE as our performance metric.

In [ ]:
# Calculate mean of results


In [ ]:
# Display a histogram of the results


## Communication of Results

Conclusion:

Future work:


